In [1]:
!pip install pandas
import pandas as pd


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
chunks = pd.read_csv("complaints.csv", chunksize=10000)

filtered = []

for chunk in chunks:
    filtered.append(chunk[chunk["Product"] == "Credit card"])

df = pd.concat(filtered)


In [3]:
print(df.sample())

        Date received      Product  \
8507361    2025-07-04  Credit card   

                                        Sub-product  \
8507361  General-purpose credit card or charge card   

                                      Issue      Sub-issue  \
8507361  Other features, terms, or problems  Other problem   

                              Consumer complaint narrative  \
8507361  On XX/XX/year>, I filed a motion requesting an...   

        Company public response                            Company State  \
8507361                     NaN  CAPITAL ONE FINANCIAL CORPORATION    AZ   

        ZIP code Tags Submitted via Date sent to company  \
8507361    85029  NaN           Web           2025-07-04   

        Company response to consumer Timely response?  Complaint ID  
8507361      Closed with explanation              Yes    14446580.0  


In [4]:
df = df[
    [
        "Consumer complaint narrative",
        "Product",
        "Issue",
        "Company",
        "Date received",
        "State"
    ]
]

In [5]:
df.describe()

,Consumer complaint narrative,Product,Issue,Company,Date received,State
count,124472,334330,334323,334330,334330,332542
unique,115308,1,49,1076,29931,63
top,See the attached documents. I want the bureau ...,Credit card,Problem with a purchase shown on your statement,"CITIBANK, N.A.",2025-01-16,CA
freq,835,334330,58354,40764,527,45128


In [6]:
new_df= df.dropna(subset=["Consumer complaint narrative"])

In [7]:
new_df.describe()

,Consumer complaint narrative,Product,Issue,Company,Date received,State
count,124472,124472,124472,124472,124472,123998
unique,115308,1,45,724,7837,61
top,See the attached documents. I want the bureau ...,Credit card,Problem with a purchase shown on your statement,"CITIBANK, N.A.",2025-12-02,CA
freq,835,124472,27550,15402,221,17270


In [8]:
# new_df.memory_usage(deep=True).sum() / (1024**2)

In [9]:
duplicates = new_df.duplicated(
    subset=["Consumer complaint narrative"]
).sum()
print(duplicates)

9164


In [10]:
complaint_counts=(
 new_df.groupby("Consumer complaint narrative").size().reset_index(name="frequency")
)

In [11]:
new_df["words_count"]=(
    new_df["Consumer complaint narrative"].str.split().str.len()
)
new_df["words_count"].describe()

C:\Users\anshu\AppData\Local\Temp\ipykernel_7044\3894858424.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["words_count"]=(


count    124472.000000
mean        210.291817
std         219.436815
min           2.000000
25%          75.000000
50%         158.000000
75%         272.000000
max        6469.000000
Name: words_count, dtype: float64

In [12]:
pd.set_option("display.max_colwidth", None)
print(new_df["Consumer complaint narrative"].sample(20,random_state=42))

8285308                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [13]:
new_df["clean_text"] = (
    new_df["Consumer complaint narrative"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

C:\Users\anshu\AppData\Local\Temp\ipykernel_7044\2611869757.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["clean_text"] = (


In [14]:
idx = 10

print("Original:\n")
print(new_df.iloc[idx]["Consumer complaint narrative"])

print("\nCleaned:\n")
print(new_df.iloc[idx]["clean_text"])

Original:

On XX/XX/year> I went online to XXXX XXXX to purchase XXXX round trip tickets to XXXX XXXX. The total cost for the TWO tickets was {$2800.00}. I went through the XXXX payment link and used my Citi MasterCard to purchase the tickets. The purchase was declined by Citi Card because they wanted to verify that it was me making the purchase. I verified it was me and Citi alerted me that the purchase could be made. I tried again and it was declined. I contacted Citi again and they said it was all set to book. 

The next day, on XX/XX/year> I went back online to XXXX XXXX to book the flights. The purchase was approved. 

In XXXX, I received my Citi Card XXXX statement, and it showed TWO PURCHASES for XXXX XXXX for the exact same flights. I was charged TWICE for the ticket purchase. I contacted Citi Card IMMEDIATELY and they agreed that there was a duplicate charge on my credit card for the ticket purchase - one for {$2800.00} on XX/XX/XXXX ( the one they initially declined ) and one

In [15]:
new_df = new_df.reset_index(drop=True)

In [16]:
new_df.sample(5)

,Consumer complaint narrative,Product,Issue,Company,Date received,State,words_count,clean_text
101100,"On XX/XX/year>, my Robinhood Gold Card account was closed by the issuer for alleged business-type transactions. I received an email titled Notice about your Robinhood Gold Card account on the same date.\n\nI do not operate a business and all transactions on the account were for personal use. After receiving the notice, I contacted Robinhood through their in-app chat support to request clarification. \n\nSpecifically, I requested : Identification of the transactions that were classified as business-type The criteria used to determine that classification Confirmation of whether the decision was automated or manually reviewed Reconsideration of the account closure Robinhood declined to provide transaction-level details or the standards used in their determination. I was informed that reinstatement requests are not accepted under any circumstances. Additionally, all remaining rewards points were forfeited as a result of the closure. \n\nAt this time, I have not been provided any specific evidence or explanation showing how my transactions violated the Cardholder Agreement. \n\nI am requesting : Identification of the specific transactions deemed business-related The criteria used to classify those transactions Confirmation of whether this determination was automated or manually reviewed A fair reconsideration of the account closure I am seeking transparency and a documented basis for this decision.",Credit card,Closing your account,ROBINHOOD MARKETS INC.,2026-02-25,NY,203,"on xx/xx/year>, my robinhood gold card account was closed by the issuer for alleged business-type transactions. i received an email titled notice about your robinhood gold card account on the same date. i do not operate a business and all transactions on the account were for personal use. after receiving the notice, i contacted robinhood through their in-app chat support to request clarification. specifically, i requested : identification of the transactions that were classified as business-type the criteria used to determine that classification confirmation of whether the decision was automated or manually reviewed reconsideration of the account closure robinhood declined to provide transaction-level details or the standards used in their determination. i was informed that reinstatement requests are not accepted under any circumstances. additionally, all remaining rewards points were forfeited as a result of the closure. at this time, i have not been provided any specific evidence or explanation showing how my transactions violated the cardholder agreement. i am requesting : identification of the specific transactions deemed business-related the criteria used to classify those transactions confirmation of whether this determination was automated or manually reviewed a fair reconsideration of the account closure i am seeking transparency and a documented basis for this decision."
93855,"This letter concerns the transaction below made with my Citi XXXX XXXX XXXX : XXXX * XXXXXXXX XXXX XXXX XXXX XXXX XXXXXXXX CA {$110.00} on XX/XX/XXXX I opened a case with the card issuer, Citibank, XXXX XX/XX/XXXX and the Case number is XXXX. The reason for Dispute was that Goods/ services are not as expected. \n\nOn XX/XX/XXXX I purchased a ticket to an event called XXXXXXXX XXXX XXXXXXXX XXXXXXXX XXXX XXXX XXXX XXXX XXXX. The event was to take place on XX/XX/XXXX. \n\nHowever, the merchant has significantly changed the details of the event. The finalized event was not what I expected. The merchant has changed the following : - The artist XXXX is no longer performing at this event. This is the main reason why I purchased this ticket - The event has moved outdoors. I can not be outdoors that long due to health concerns. \n\nBecause of these changes, I requested a refund from the merchant, but with no response. Because of the event not having the expected outcome, I have filed a d

In [17]:
!pip install sentence-transformers


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
dev_df = new_df.sample(5000, random_state=42).reset_index(drop=True)

In [ ]:
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("all-MiniLM-L6-v2")
# embeddings=model.encode(
#     dev_df["clean_text"].tolist(),
#     batch_size=32,
#     show_progress_bar=True,
#     convert_to_numpy=True
# )

In [27]:
import numpy as np

embeddings = np.load("embeddings.npy")

In [ ]:
# np.save("embeddings.npy",embeddings)

In [22]:
pip install hdbscan umap-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


768-D embeddings
        ↓
UMAP (50 dimensions)
        ↓
HDBSCAN

In [23]:
#why to use umap
#faster clustring,less memory,often better cluster

In [28]:
import umap
reducer = umap.UMAP(
    n_neighbors=15,
    n_components=50,
    metric="cosine",
    random_state=42
)
reduced_embeddings=reducer.fit_transform(embeddings)

c:\Users\anshu\OneDrive\Desktop\project\feedback\fee\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [29]:
print(reduced_embeddings.shape)

(5000, 50)


In [30]:
import hdbscan
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    metric="euclidean",
    prediction_data=True
)
labels = clusterer.fit_predict(reduced_embeddings)

In [31]:
import numpy as np

print("Clusters:", len(np.unique(labels)) - (1 if -1 in labels else 0))
print("Noise:", np.sum(labels == -1))

Clusters: 32
Noise: 1851


In [32]:
dev_df["cluster"] = labels

In [33]:
cluster_5 = dev_df[dev_df["cluster"] == 5]

cluster_5["clean_text"].head(10)

78                                                                 i ensure that i always pay on time, and i'm certain of this fact. what i don't understand is why this company is reporting late payments for certain months. per usc 1666b, billing errors need to be corrected, or they owe me {$1000.00} for each inaccurately reported account.
138     timely payments are always a priority for me, and i am certain about this. however, i am unsure why this company is reporting me as late in certain months, which should not be the case. according to usc 1666b, any billing error should be corrected, or they will be liable to pay me {$1000.00} for each account reporting inaccurately.
182                                                                i ensure that i always pay on time, and i'm certain of this fact. what i don't understand is why this company is reporting late payments for certain months. per usc 1666b, billing errors need to be corrected, or they owe me {$1000.00} for each inacc

In [34]:
import numpy as np

clusters = np.unique(labels)

print("Total Clusters:", len(clusters) - (1 if -1 in clusters else 0))
print("Noise:", np.sum(labels == -1))

Total Clusters: 32
Noise: 1851


In [35]:
cluster_sizes = (
    dev_df["cluster"]
    .value_counts()
    .sort_values(ascending=False)
)

print(cluster_sizes)

cluster
-1     1851
 13     693
 16     289
 24     242
 31     173
 21     163
 3      146
 20     129
 25     114
 17     105
 14     102
 8      102
 11      98
 23      87
 12      71
 0       57
 15      54
 27      49
 18      43
 7       41
 9       40
 28      38
 5       36
 2       33
 6       32
 19      30
 22      30
 10      29
 29      27
 26      26
 4       24
 1       23
 30      23
Name: count, dtype: int64


In [36]:
largest_cluster = cluster_sizes.index[0]

print("Largest cluster:", largest_cluster)

samples = dev_df[dev_df["cluster"] == largest_cluster]["clean_text"].head(10)

for i, complaint in enumerate(samples, 1):
    print(f"\nComplaint {i}")
    print(complaint)
    print("-" * 80)

Largest cluster: -1

Complaint 1
xxxx i've been a card holder for over xxxx years and my daughter has been a card holder for over xxxx years. we recently experienced our accounts being merged. my daughter purchase are being applied to my account causing my balance to go extremely high. it's impacting my xxxx payment, and when i call wich has been multiple times they canceled my account without my consent internally to prevent further billing. but this action is a direct impact to my credit. all i wanted is to have my account corrected to reflect the ture purchase. the supervisor wasn't much help, she never addressed any of the notes from her colleagues addressing the known issue her name was xxxx xxxx xxxx. the bottom line, they are billing me for my purchases and for my daughter purchase. i appreciate your attention to this unlawful matter. thanks
--------------------------------------------------------------------------------

Complaint 2
i have had a store credit card with target si

In [37]:
largest_cluster = cluster_sizes[cluster_sizes.index != -1].index[0]

In [38]:
largest_cluster = cluster_sizes.index[0]

print("Largest cluster:", largest_cluster)

samples = dev_df[dev_df["cluster"] == largest_cluster]["clean_text"].head(10)

for i, complaint in enumerate(samples, 1):
    print(f"\nComplaint {i}")
    print(complaint)
    print("-" * 80)

Largest cluster: -1

Complaint 1
xxxx i've been a card holder for over xxxx years and my daughter has been a card holder for over xxxx years. we recently experienced our accounts being merged. my daughter purchase are being applied to my account causing my balance to go extremely high. it's impacting my xxxx payment, and when i call wich has been multiple times they canceled my account without my consent internally to prevent further billing. but this action is a direct impact to my credit. all i wanted is to have my account corrected to reflect the ture purchase. the supervisor wasn't much help, she never addressed any of the notes from her colleagues addressing the known issue her name was xxxx xxxx xxxx. the bottom line, they are billing me for my purchases and for my daughter purchase. i appreciate your attention to this unlawful matter. thanks
--------------------------------------------------------------------------------

Complaint 2
i have had a store credit card with target si

In [39]:
cluster_sizes = (
    dev_df["cluster"]
    .value_counts()
    .sort_values(ascending=False)
)

print(cluster_sizes)

cluster
-1     1851
 13     693
 16     289
 24     242
 31     173
 21     163
 3      146
 20     129
 25     114
 17     105
 14     102
 8      102
 11      98
 23      87
 12      71
 0       57
 15      54
 27      49
 18      43
 7       41
 9       40
 28      38
 5       36
 2       33
 6       32
 19      30
 22      30
 10      29
 29      27
 26      26
 4       24
 1       23
 30      23
Name: count, dtype: int64


In [40]:
largest_cluster = cluster_sizes[cluster_sizes.index != -1].index[0]

print(largest_cluster)

13


In [41]:
largest_cluster = cluster_sizes[cluster_sizes.index != -1].index[0]

print(largest_cluster)

13


In [42]:
samples = dev_df[
    dev_df["cluster"] == largest_cluster
]["clean_text"]

for i, complaint in enumerate(samples.head(10), 1):
    print(f"\nComplaint {i}")
    print(complaint)
    print("-"*80)


Complaint 1
i am writing to formally file a complaint regarding an account listed in your system that utilizes my residential address. on or around xx/xx/xxxx, i received a letter from american express dated xx/xx/xxxx. the letter was addressed to xxxx xxxx and referenced an account ending in xxxx. the correspondence confirmed the cancellation of said account. xxxx xxxx xxxx does not reside at this address ( xxxx xxxx xxxx, xxxx xxxx, nj ). this individual has no legal right to use this property for billing or correspondence. the association of this individual with my residence suggests either a clerical error in your database or potential application fraud where my address was used without my consent. i request that you immediately take the following actions : 1. remove my address ( xxxx xxxx xxxx, xxxx xxxx, nj xxxx ) from the profile associated with xxxx xxxx and account # xxxx immediately. 2. cease all future correspondence addressed to this individual at my home. 3. investigate t

In [43]:
cluster_summary = (
    dev_df.groupby("cluster")
    .agg(
        complaints=("clean_text", "count"),
        unique_issues=("Issue", "nunique")
    )
    .sort_values("complaints", ascending=False)
)

print(cluster_summary.head(10))

         complaints  unique_issues
cluster                           
-1             1851             42
 13             693             35
 16             289             26
 24             242             29
 31             173             18
 21             163             18
 3              146             11
 20             129             16
 25             114              7
 17             105             24


In [44]:
cluster = 13

samples = dev_df[
    dev_df["cluster"] == cluster
][["Issue", "clean_text"]].sample(10, random_state=42)

for _, row in samples.iterrows():
    print("Issue:", row["Issue"])
    print(row["clean_text"])
    print("-" * 100)

Issue: Balance transfer
in 2016 american express ended its role as the exclusive card for xxxx, with that role being taken over by xxxx. my existing amex/xxxx card had a balance at the time of the change and that balance was transfered to the xxxx account. subsequently, amex mailed me an offer to open a new account with them ; the offer extended stated that balance transfers and purchases would be interest free for 15 months. taking advantage of this i transferred the xxxx balance ( the original amex/costco balance ) back to amex under the new, interest free offer. however, in the first billing cycle, amex charged me interest on the balance transfer made, contrary to their offer terms under which i made said transfer. i called amex and opened a complaint. after a month of not hearing anything i called amex and inquired of the status. i was told a decision was forthcoming. i waited another 4 weeks and phoned again. during this xxxx call amex made the assertion that the interest free off

In [45]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

representatives = {}

for cluster in sorted(dev_df["cluster"].unique()):
    if cluster == -1:
        continue

    indices = dev_df[dev_df["cluster"] == cluster].index

    cluster_embeddings = embeddings[indices]

    centroid = cluster_embeddings.mean(axis=0)

    similarities = cosine_similarity(
        cluster_embeddings,
        centroid.reshape(1, -1)
    ).flatten()

    representative_index = indices[similarities.argmax()]

    representatives[cluster] = representative_index

In [46]:
cluster = 13

idx = representatives[cluster]

print(dev_df.loc[idx, "clean_text"])

hello- i have been a chase united cardholder since xxxx. in xx/xx/xxxx, i was unexpectedly informed that my credit card account had been permanently closed. i only discovered this after my card was declined during a transaction, and i called customer service. at that time, representatives provided limited information and stated that a letter would follow with more details. however, the letter contained no additional explanationonly confirmation of the account closure. at the time of closure, my account had a {$0.00} balance, with all payments fully current. i had several upcoming travel plans booked on the card, including flights and a xxxx trip, which were canceled because chase flagged them as fraudulent and reversed the charges. this caused my xxxx tickets to be rescinded, requiring me to repurchase them out of pocket. i have not yet been reimbursed for this cost, despite multiple follow-ups. since the week of xx/xx/xxxx, ive contacted chase several times, but each representative ha

In [47]:
cluster_report = (
    dev_df[dev_df["cluster"] != -1]
    .groupby("cluster")
    .agg(
        complaints=("clean_text", "count"),
        top_company=("Company", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        top_issue=("Issue", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    )
)

cluster_report.head()

,complaints,top_company,top_issue
cluster,,,
0,57,"EQUIFAX, INC.",Problem with a company's investigation into an existing problem
1,23,"EQUIFAX, INC.",Problem with a company's investigation into an existing problem
2,33,"EQUIFAX, INC.",Incorrect information on your report
3,146,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",Problem with a company's investigation into an existing problem
4,24,"EQUIFAX, INC.",Incorrect information on your report
